# Lesson 04 - Tool Use Design Pattern

For dis lesson you go learn di **Tool Use** design pattern for AI agents wey dey use di Microsoft Agent Framework (Python). We go cover:

- How to define function tools wit di `@tool` decorator and typed parameters
- How to provide tool schemas so di model sabi wetin each tool dey do
- How to control tool execution wit `approval_mode`
- How to return **structured output** using Pydantic models and `response_format`

Di scenario na **travel booking agent** wey fit find destinations, check availability, and collect flight information.


## Setup


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Defining Tools with the @tool Decorator

The `@tool` decorator dey turn plain Python function to tool wey agent fit call.
Key points:

- The **docstring** na the tool description wey model go see.
- **Type annotations** (including `Annotated` wey get descriptions) na how tool schema dem take define.
- `approval_mode` na to control if user must approve each call before e run.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Creating an Agent with Multiple Tools

Pass all di three tools to di client so di model fit invoke any wey e need to answer di user's question.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Structured Output wit Tools

By setting `response_format` to a Pydantic model, di agent dey forced to return one well-typed JSON object instead of free-form text. Dis one dey useful wen downstream code need to use di result programmatically.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Tool Approval Patterns

Di `approval_mode` parameter wey dey for `@tool` dey control whether tool calls need human approval before dem go run:

| Mode | Behaviour |
|---|---|
| `"never_require"` | Tool go run by itself — no need make user approve am. |
| `"always_require"` | Every call must get user approval before e go run. |

Make you use `"always_require"` for tools wey get side-effects (like booking flight, charging credit card) so human fit dey involved.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

For dis lesson you don learn how to:

1. **Define tools** wit di `@tool` decorator wey get typed parameters and docstrings wey dey serve as di tool schema.
2. **Compose multiple tools** so di agent fit call dem one by one to answer complex queries.
3. **Return structured output** by passing Pydantic model as `response_format`.
4. **Control tool approval** wit `approval_mode` to keep human dey involved for sensitive operations.

Dis patterns na di foundation for building reliable, production-ready agents wey fit interact wit external systems safely.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we try make e correct, abeg sabi say automated translation fit get errors or mistakes. Di original document for e own language na di correct source. For important info, better make human professional translate am. We no go take responsibility for any misunderstanding or wrong meaning wey fit show from dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
